# Clustering project - Tymoteusz Bultrowicz

## Setup
First cell is used to set up a new kernel for notebook. It requires a "requirements.txt" text file attached to the solution to run properly.

In [ ]:
import os
import sys
import subprocess

def setup_environment():
    IN_COLAB = 'google.colab' in sys.modules
    
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
        print("Installation successful.")
    else:
        venv_dir = "venv"
        
        if not os.path.exists(venv_dir):
            subprocess.run([sys.executable, "-m", "venv", venv_dir], check=True)
        
        python_executable = os.path.join(venv_dir, "bin", "python")
        if os.path.exists("requirements.txt"):
            subprocess.run([python_executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
            subprocess.run([python_executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
        else:
            print("Please provide requirements.txt attached!")

        print("Installation successful. Remember to select correct Python interpreter in your IDE.")

setup_environment()


  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
  Using cached ipykernel-7.2.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached optuna-4.8.0-py3-none-any.whl.metadata (17 kB)
  Using cached umap_learn-0.5.12-py3-none-any.whl.metadata (24 kB)
  Using cached comm-0.2.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached ipython-9.13.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached jupyter_client-8.8.0-py3-none-any.whl.metadata (8.4 kB)
  Using cached jupyter_core-5.9.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached matplotlib_inline-0.2.2-py3-none-any.whl.metadata (2.4 kB)
  Using cached nest_asyncio-1.6.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached packaging-26.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached psuti

### Bash equivalent - paste into console:
python3 -m venv venv \
venv/bin/python -m pip install --upgrade pip \
venv/bin/python -m pip install -r requirements.txt

### Input - please set these global variables accordingly to your usage.

**Important!** When trying out train mode, you need to save training set included in exercise as **directory "training_samples"** and besides that put input.txt & labels.txt in a working directory.

For test mode - the one that does the clustering, only thing needed is proper dir_file_path variable.

In [ ]:
# File containing the image paths for the clustering task
dir_file_path = "inputtt.txt" 

# Mode - train/test. Train recreates the process of finding good model for clustering task.
# Test - default mode, does the clustering using the best model found in offline train mode.
mode = "test"

### Imports and constants' definitions

In [ ]:
import os
import warnings
import time
import numpy as np
import optuna as opt
from PIL import Image
from umap import UMAP
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans, AgglomerativeClustering, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from skimage.feature import hog

# Supressing anoying warnings...
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

RAND = 42 # Random seed for reproducibility
STD_SIZE = (128, 128) # Size to which we resize the images
NUM_TRIALS = 250 # Number of trials for hyperparameter optimization in optuna.

# Different combinations of parameters for HOG feature extraction
HOG_PARAMS = [
    (4, 1),
    (4, 2),
    (8, 1),
    (8, 2),
    (8, 4),
    (16, 1),
    (16, 2),
    (16, 4)
]

# And string representations which can be passed to optuna.
HOG_STR_PARAMS = [f"{pcc}_{cpb}" for pcc, cpb in HOG_PARAMS] 

# Parameters for dimensionality reduction.
KEEP_96_VARIANCE = 0.96
# Numbers of cluster which our algorithms will try to find.
K_MIN_SIZE = 2
K_MAX_SIZE = 60
# Number of initializations for KMeans.
N_INIT = 3

UMAP_PARAMS = {
    'n_neighs_min' : 15,
    'n_neighs_max' : 50,
    'n_components_min': 2,
    'n_components_max' : 10,
    'min_dist' : 0.0,
}
HDBSCAN_PARAMS = {
    'min_cluster_size_min': 5,
    'min_cluster_size_max': 50,
    'cluster_selection_epsilon_min': 0.0,
    'cluster_selection_epsilon_max': 0.8,
}

### Getting input data.

We use pillow, then remove two of R,G,B columns (as images are greyscale). Finally we flatten images, so single image is a row of 128x128 pixels.

In [ ]:
images = []
image_names = []
print(f"Processing direction file at: {dir_file_path}")

with open(dir_file_path, 'r') as dir_file:
    for line in dir_file:
        file = line.strip()
        image_names.append(file)
        with Image.open(file) as image_file:
            img = image_file.copy()

        img = img.convert('RGB').resize(STD_SIZE)
        images.append(np.array(img)[:, :, :1].reshape(-1))

Processing direction file at: input.txt


## My approach
There are couple of challenges we need to go through, to properly classify pictures. 

1. Unifying sizes - due to fact, that some pictures were bigger and I didn't want to squeeze them, I used resizing to 128x128 pixels size.

2. Extracting photo features - I wanted to see if using any algorithm from scikit-image library will help clustering algorithms notice important details. For that I wanted to try out the Histogram of Gradients - HOG algorithm. In training I also tested how skipping these phase affects clustering.

**Why HOG?** - Converting images into vectors of gradients helps to reduce effects of shifting. Besides that it points out more differences between very similar letters, that have different kind of curvature, edges etc.

3. Dimensionality reduction - because every photo would now had 16384 (128 x 128) columns, I needed to find good algorithm for reducing size of data. Raw data would make algorithms run too slow and good results would be rare due to curse of multidimensiality. During training I considered:

**PCA** - canonical, awesome tool for reducing dimensionality while preserving lots of information about the data. \
**UMAP** - due to it's interesting principle of keeping close points still together using non-linear transformation, I thought it could be good alternative to PCA, working well with algoritms like HDBSCAN.

4. Mysterious k - Because we don't know the correct amount of clusters, I needed to choose between algorithms that pick amount of clusters by themselves (HDBSCAN). I also wanted however too be available to use algorithms with a given amount of clusters, for that I designed an approach of a hybrid model, which uses a loop with some k values and returns the one with the best result.

5. Scoring metric - as we don't know the labels of images given by some 'correct' clustering mechanism, we are only left with score metrics like silhoutte_score, caliniski-harabasz index etc. and some variations on those. After some experiments with visual judging of effects I decided to work with classical silhoutte_score.

### Summary:
Our algorithm can be described as:

Load pictures --> rescale --> extract HOG features --> scaling --> reduce dimentions --> cluster --
  --> Return labels, generate results.


And for clustering: \
If algorithm doesn't require n_components:   run it --> collect results
If algorithm does require n_components:  
  1. Run loops with different k values --> \
  2. Score them with silhouette score using nearly original data --> \
  3. Return labels of the best rated clustering 

## Train mode - picking what works best.

For finding correct combination of preprocessing techniques and final clustering algorithm I decided to use optuna. 

To have high quality measure of how models behave, I prepared labels for 387 photos miniset, to compare with using ARI.
(given input.txt and labels inside labels.txt)

Locally whole training process worked around 10-12 minutes. Saves results in best_params.txt


In [ ]:
# Implementing HOG feature extraction for all images with given parameters.
# HOG algorithm transforms each image into a feature vector which captures the 
# distribution of gradients and orientations in the image.

def hog_transform(images, pcc, cpb):
    hog_images = []
    for image in images:
        hog_image = hog(
            image.reshape(STD_SIZE), 
            orientations = 8,
            pixels_per_cell=pcc, 
            cells_per_block=cpb,
            feature_vector=True
        )
        hog_images.append(hog_image)
    return np.array(hog_images)

In [ ]:
# Objective function for optuna optimization.
# This function represents single trial of hyperparameter optimization.
def objective(trial, X, correct_labels):
    # Whether to use HOG features or not.
    use_hog = trial.suggest_categorical('use_hog', [True, False])
    if use_hog:
        str = trial.suggest_categorical('hog_params', HOG_STR_PARAMS)
        pcc, cpb = map(int, str.split('_'))
        X = hog_transform(X, (pcc, pcc), (cpb, cpb))

    # Standardizing the features before dimensionality reduction.
    X = StandardScaler().fit_transform(X)

    # Whether to use PCA or UMAP for dimensionality reduction.
    pca_umap = trial.suggest_categorical('pca_umap', ['PCA', 'UMAP'])
    if pca_umap == 'UMAP':
        # Picking UMAP hyper parameters.
        n_components = trial.suggest_int('umap_n_components', UMAP_PARAMS['n_components_min'], UMAP_PARAMS['n_components_max'])
        n_neighbors = trial.suggest_int('umap_n_neighbors', UMAP_PARAMS['n_neighs_min'], UMAP_PARAMS['n_neighs_max'])
        X = UMAP(n_components=n_components, n_neighbors=n_neighbors, min_dist=UMAP_PARAMS['min_dist']).fit_transform(X)
    else:
       X = PCA(n_components=KEEP_96_VARIANCE, random_state=RAND).fit_transform(X) 

    # Picking model for clustering.
    model = trial.suggest_categorical('model', ['KMeans', 'AgglomerativeClustering', 'GaussianMixture', 'HDBSCAN'])
    if model == 'GaussianMixture':
        # For gaussian mixture we also pick the covariance type.
        covariance_type = trial.suggest_categorical('covariance_type', ['tied', 'diag', 'spherical'])
    elif model == 'HDBSCAN':
        # For HDBSCAN we pick it's params and avoid extra loop.
        min_cluster_size = trial.suggest_int(
            'min_cluster_size', 
            HDBSCAN_PARAMS['min_cluster_size_min'], 
            HDBSCAN_PARAMS['min_cluster_size_max']
        )
        cluster_selection_epsilon = trial.suggest_float(
            'cluster_selection_epsilon', 
            HDBSCAN_PARAMS['cluster_selection_epsilon_min'], 
            HDBSCAN_PARAMS['cluster_selection_epsilon_max']
        )

        # As HDBSCAN with bad parameters combo can throw errors.
        try:
            mdl = HDBSCAN(min_cluster_size=min_cluster_size, cluster_selection_epsilon=cluster_selection_epsilon)
            labels = mdl.fit_predict(X)
        except Exception as e:
            return -1

        return adjusted_rand_score(labels, correct_labels)
    
    best_score = -1
    best_labels = None
    # In loop we try different numbers of clusters.
    # Due to fact that in testing we cannot access labels, we evaluate best k
    # using silhoutte score and then we return ARI score for best labels as trail result.
    for k in range(K_MIN_SIZE, K_MAX_SIZE + 1):
        if model == 'KMeans':
            clustering = KMeans(n_clusters=k, random_state=RAND, n_init=N_INIT).fit(X)
            labels = clustering.labels_
        elif model == 'AgglomerativeClustering':
            clustering = AgglomerativeClustering(n_clusters=k, linkage='ward', metric='euclidean').fit(X)
            labels = clustering.labels_
        # GaussianMixture - it can fail with improper ks.
        elif model == 'GaussianMixture' and k <= X.shape[1]:
            clustering = GaussianMixture(n_components=k, covariance_type=covariance_type, random_state=RAND).fit(X)
            labels = clustering.predict(X)
        else:
            continue
        # Sanity check - to avoid errors thrown by silhouette_score.
        unique = np.unique(labels)
        if len(unique) < 2 or len(unique) >= len(X):
            continue

        score = silhouette_score(X, labels)
        if score > best_score:
            best_score = score
            best_labels = labels

    return adjusted_rand_score(best_labels, correct_labels)

# Function conducting optuna optimization study.
def train_mode(images, correct_labels):
    opt.logging.set_verbosity(opt.logging.WARNING)
    study = opt.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, images, correct_labels), n_trials=NUM_TRIALS, show_progress_bar=True)
    return study


if mode == "train":
    correct_labels = []
    with open('labels.txt', 'r') as f:
        for line in f:
            label = line.strip()
            correct_labels.append(label)

    study = train_mode(images, correct_labels)
    with open('best_params.txt', 'w') as f:
        f.write(str(study.best_params))
        f.write(f"\nBest Adjusted Rand Score: {study.best_value:.4f}\n")
    print("Study done. Results saved to best_params.txt")

### Testing phase

For test mode we simply replicate single trial from optuna. We pick best results achieved during my programme execution, it was:

**Use HOG**: True, pixels_per_cell = (16,16), cells_per_block = (2, 2) \
**PCA / UMAP**: PCA() \
**Model**: Agglomerative Clustering. \
**Metric**: One of most important hyper parameters. From training phase it was made visible that default euclidean matric in ward linkage works the best for our task.

For PCA() I've ran across curse of multidimentionality problem. When I set up PCA for preserving 96% of variance, amount of columns on big datasets was so big, that distances between images started to fade, euclidean metric started having problems with proper clustering.

My fix for that was to set **n_components** to constant number, for which I found **80** to work well enough.

**RUN TIME:** script should run no more than ~2-3 minutes for database of 7600 pictures. 


In [ ]:
### Helper functions to save the results in proper format.
def pretty_print(names):
    return " ".join([os.path.basename(name) for name in names])

def make_dict(image_names, labels):
    result = {}
    for name, label in zip(image_names, labels):
        if label not in result:
            result[label] = []
        result[label].append(name)
    return result

def generate_txt(result, output_file='output_clusters.txt'):
    with open(output_file, 'w', newline='\n') as f:
        for names in result.values():
            f.write(pretty_print(names) + '\n')

def generate_html(result, output_file='output_clusters_images.html'):
    with open(output_file, 'w') as f:
        f.write("<html><body>\n")
        for names in result.values():
            f.write("<div>\n")
            for name in names:
                f.write(f"<img src='{name}' alt='{name}' style='width:128px;height:128px;'>\n")
            f.write("</div>\n")
            f.write("<hr>\n")
        f.write("</body></html>\n")

In [ ]:
# Main pipeline of clustering task, from preprocessing to clustering and saving the results.
def cluster_images(images):
    print("Staring to preprocess images ...")
    n_data = len(images)
    scaler = StandardScaler()
    pca = PCA(n_components=min(n_data - 1, 80))

    judge_images = images.copy()
    judge_images = scaler.fit_transform(judge_images)
    judge_images = pca.fit_transform(judge_images)

    data_img = hog_transform(images, (16, 16), (2, 2))
    print("Hog transform done, starting dimensionality reduction ...")
    data_img = scaler.fit_transform(data_img)
    data_img = pca.fit_transform(data_img)

    print("Beginning clustering for different k values ...")
    best_labels = None
    best_score = -1
    for k in range(K_MIN_SIZE, min(K_MAX_SIZE + 1, n_data)):
        clustering = AgglomerativeClustering(n_clusters=k, linkage='ward', metric='euclidean').fit(data_img)
        labels = clustering.labels_
        score = silhouette_score(judge_images, labels)
        if score > best_score:
            best_score = score
            best_labels = labels

        if k % 10 == 0:
            print(f"Done for {k} out of {min(K_MAX_SIZE, n_data)} k values.")
    return best_labels

if mode == "test":
    start = time.time()
    labels = cluster_images(images)
    result = make_dict(image_names, labels)

    generate_txt(result)
    generate_html(result)
    elapsed = time.time() - start
    print(f"Clustering complete. Results saved to output_clusters.txt and output_clusters_images.html. Time taken: {elapsed:.1f}s")

Staring to preprocess images ...
Hog transform done, starting dimensionality reduction ...
Beginning clustering for different k values ...
Done for 10 out of 60 k values.
Done for 20 out of 60 k values.
Done for 30 out of 60 k values.
Done for 40 out of 60 k values.
Done for 50 out of 60 k values.
Done for 60 out of 60 k values.
Clustering complete. Results saved to output_clusters.txt and output_clusters_images.html. Time taken: 3.5s


### Algorithms used - sources:

Working on PNGs - pillow IMAGE module:
https://pillow.readthedocs.io/en/stable/reference/Image.html

Clustering algorithms come from scikit-learn cluster packet: \
https://scikit-learn.org/stable/api/sklearn.cluster.html \
https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html

PCA algorithm:
https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

Scaling algorithm:
https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

UMAP algorithm:
https://umap-learn.readthedocs.io/en/latest/index.html

Metric algorithm adjusted_rand_score & silhouette_score come from library:
https://scikit-learn.org/stable/api/sklearn.metrics.html

Histogram of Gradients - HOG:
https://scikit-image.org/docs/stable/api/skimage.feature.html#skimage.feature.hog
